# Generate BloodMNIST predictions: ResNet-18 and ViT

Run the cells in order using an environment with the repository's `requirements.txt` installed (plus Jupyter).
All weights and data are loaded locally; nothing is downloaded or trained.

Outputs (relative to the repository root):
```
preds/
  resnet/  # train.csv, val.csv, test.csv, matching .npz files, metadata.json
  vit/     # same schema and sample order
  summary.json
```
Each CSV contains the split-local sample index, true/predicted labels and names, correctness,
confidence, and eight class probabilities. NPZ files also contain the raw logits.
The same sample index identifies the same image for both models within a split.

**Models:** official MedMNIST `resnet18_28_1.pth` and the local fine-tuned ViT.
ResNet uses normalized 28×28 RGB inputs; ViT uses its supplied processor to resize to 224×224.
This is not a reproduction of the ViT publisher's benchmark; the source dataset here is 28×28.
For research/teaching only, not clinical use.


In [ ]:
from pathlib import Path
import csv
import gc
import json
import sys
import time

import numpy as np
import torch
from PIL import Image
from transformers import ViTForImageClassification, ViTImageProcessor

# Works when launched from the repository root or its skript/ directory.
# Set ROOT explicitly if your Jupyter server uses a different working directory.
ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "data/metadata.json").is_file() and (p / "models/vit/config.json").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Set ROOT to the repository directory before continuing.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from models.resnet18.models import ResNet18

SPLITS = ("train", "val", "test")
BATCH_SIZE = 32  # Reduce to 8 or 16 if ViT runs out of memory.
MAX_SAMPLES = None  # Set to 8 for a smoke test; restore None for full exports.
OVERWRITE = False  # Set True explicitly to replace existing exports.
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
OUT = ROOT / "preds" if MAX_SAMPLES is None else ROOT / "preds_smoke"
print(f"Repository: {ROOT}\nDevice: {DEVICE}\nOutput: {OUT}")


## Load data and validate local assets
Class IDs follow `data/metadata.json`. Models are processed one at a time to limit memory usage.
Smoke-test exports go to `preds_smoke/` so they cannot be mistaken for full-dataset predictions.


In [ ]:
required = [
    "data/bloodmnist.npz", "models/resnet18/resnet18_28_1.pth",
    "models/vit/model.safetensors", "models/vit/config.json",
    "models/vit/preprocessor_config.json",
]
for relative in required:
    path = ROOT / relative
    if not path.is_file():
        raise FileNotFoundError(path)
    with path.open("rb") as handle:
        if handle.read(100).startswith(b"version https://git-lfs.github.com/spec"):
            raise RuntimeError(f"{relative} is a Git LFS pointer. Run `git lfs pull` first.")

metadata = json.loads((ROOT / "data/metadata.json").read_text())
class_names = [metadata["label"][str(i)] for i in range(len(metadata["label"]))]
NUM_CLASSES = len(class_names)
if BATCH_SIZE < 1 or (MAX_SAMPLES is not None and MAX_SAMPLES < 1):
    raise ValueError("BATCH_SIZE and MAX_SAMPLES must be positive.")

data = {}
with np.load(ROOT / "data/bloodmnist.npz", allow_pickle=False) as archive:
    for split in SPLITS:
        images = archive[f"{split}_images"]
        labels = archive[f"{split}_labels"].reshape(-1).astype(np.int64)
        assert images.dtype == np.uint8 and images.shape[1:] == (28, 28, 3)
        assert len(images) == len(labels) and len(images) > 0
        assert ((labels >= 0) & (labels < NUM_CLASSES)).all()
        if MAX_SAMPLES is not None:
            images, labels = images[:MAX_SAMPLES], labels[:MAX_SAMPLES]
        data[split] = (images, labels)
        print(split, images.shape, labels.shape)

# Fail before inference rather than silently overwrite expensive previous results.
for family in ("resnet", "vit"):
    directory = OUT / family
    for name in ["metadata.json", *[f"{s}.{ext}" for s in SPLITS for ext in ("csv", "npz")]]:
        if (directory / name).exists() and not OVERWRITE:
            raise FileExistsError(f"{directory / name}: set OVERWRITE=True to replace exports.")
if (OUT / "summary.json").exists() and not OVERWRITE:
    raise FileExistsError(f"{OUT / 'summary.json'} already exists.")


## Inference and export helpers
For logits $z_c$, the saved probabilities are $p_c=\exp(z_c)/\sum_j\exp(z_j)$.
The predicted label is $rg\max_c p_c$ and confidence is $\max_c p_c$.
CSV and NPZ rows preserve the original archive order; there is no shuffling or augmentation.


In [ ]:
def load_model(family):
    if family == "resnet":
        model = ResNet18(in_channels=3, num_classes=NUM_CLASSES)
        checkpoint = torch.load(
            ROOT / "models/resnet18/resnet18_28_1.pth",
            map_location="cpu", weights_only=True,
        )
        model.load_state_dict(checkpoint["net"], strict=True)
        processor = None
    elif family == "vit":
        directory = ROOT / "models/vit"
        processor = ViTImageProcessor.from_pretrained(directory, local_files_only=True)
        model = ViTForImageClassification.from_pretrained(directory, local_files_only=True)
        if model.config.num_labels != NUM_CLASSES:
            raise ValueError("ViT output size does not match dataset classes.")
        # The local model card documents the same class order as metadata.json.
    else:
        raise ValueError(f"Unknown model family: {family}")
    return model.to(DEVICE).eval(), processor


@torch.inference_mode()
def predict(model, processor, images):
    all_logits, all_probabilities = [], []
    for start in range(0, len(images), BATCH_SIZE):
        batch = images[start:start + BATCH_SIZE]
        if processor is None:
            x = torch.from_numpy(batch.transpose(0, 3, 1, 2).copy()).float()
            x = (x / 255.0 - 0.5) / 0.5
            logits = model(x.to(DEVICE))
        else:
            inputs = processor(
                images=[Image.fromarray(image) for image in batch], return_tensors="pt"
            )
            inputs = {key: value.to(DEVICE) for key, value in inputs.items()}
            logits = model(**inputs).logits
        all_logits.append(logits.float().cpu().numpy())
        all_probabilities.append(logits.float().softmax(dim=-1).cpu().numpy())
        if start == 0 or start + BATCH_SIZE >= len(images) or (start // BATCH_SIZE) % 50 == 0:
            print(f"  {min(start + BATCH_SIZE, len(images))}/{len(images)} images", flush=True)
    logits = np.concatenate(all_logits)
    probabilities = np.concatenate(all_probabilities)
    assert probabilities.shape == (len(images), NUM_CLASSES)
    assert np.isfinite(logits).all() and np.isfinite(probabilities).all()
    np.testing.assert_allclose(probabilities.sum(axis=1), 1, atol=1e-5)
    return logits, probabilities


def export_split(family, split, labels, logits, probabilities):
    directory = OUT / family
    directory.mkdir(parents=True, exist_ok=True)
    predicted = probabilities.argmax(axis=1)
    indices = np.arange(len(labels), dtype=np.int64)
    confidence = probabilities.max(axis=1)
    correct = predicted == labels
    with (directory / f"{split}.csv").open("w", newline="", encoding="utf-8") as handle:
        writer = csv.writer(handle)
        writer.writerow([
            "sample_index", "split", "true_label", "true_class", "predicted_label",
            "predicted_class", "correct", "confidence",
            *[f"prob_{i}" for i in range(NUM_CLASSES)],
        ])
        for i in indices:
            writer.writerow([
                int(i), split, int(labels[i]), class_names[labels[i]], int(predicted[i]),
                class_names[predicted[i]], bool(correct[i]), float(confidence[i]),
                *probabilities[i].tolist(),
            ])
    np.savez_compressed(
        directory / f"{split}.npz", sample_index=indices, labels=labels,
        predicted_labels=predicted, logits=logits, probabilities=probabilities,
        confidence=confidence, correct=correct, class_names=np.asarray(class_names),
    )
    return {"n_samples": len(labels), "accuracy": float(correct.mean())}


## Generate both models' predictions
This runs inference on every configured split. ViT inference on CPU can take a while.
No accuracy is assumed in advance; all metrics below come from the current local models.


In [ ]:
summary = {}
for family in ("resnet", "vit"):
    print(f"\nLoading {family} ...", flush=True)
    started = time.perf_counter()
    model, processor = load_model(family)
    summary[family] = {}
    for split, (images, labels) in data.items():
        print(f"{family}: {split}", flush=True)
        logits, probabilities = predict(model, processor, images)
        metrics = export_split(family, split, labels, logits, probabilities)
        summary[family][split] = metrics
        print(f"  accuracy={metrics['accuracy']:.4%}")
    details = {
        "model": family,
        "checkpoint": "models/resnet18/resnet18_28_1.pth" if family == "resnet" else "models/vit",
        "dataset": "data/bloodmnist.npz", "class_names": class_names,
        "source_resolution": [28, 28], "model_input_resolution": [28, 28] if family == "resnet" else [224, 224],
        "sample_order": "original archive order; sample_index is split-local and zero-based",
        "smoke_test": MAX_SAMPLES is not None,
        "device": str(DEVICE), "torch_version": str(torch.__version__),
        "metrics": summary[family], "elapsed_seconds": time.perf_counter() - started,
    }
    (OUT / family / "metadata.json").write_text(json.dumps(details, indent=2), encoding="utf-8")
    del model, processor
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    elif DEVICE.type == "mps":
        torch.mps.empty_cache()

(OUT / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))
print(f"Saved predictions to {OUT}")


## Verify exports and compare predictions
Reloaded output files must agree on sample order and ground truth before comparing models.


In [ ]:
for split in SPLITS:
    with np.load(OUT / "resnet" / f"{split}.npz", allow_pickle=False) as r, np.load(
        OUT / "vit" / f"{split}.npz", allow_pickle=False
    ) as v:
        np.testing.assert_array_equal(r["sample_index"], v["sample_index"])
        np.testing.assert_array_equal(r["labels"], v["labels"])
        np.testing.assert_array_equal(r["labels"], data[split][1])
        agreement = np.mean(r["predicted_labels"] == v["predicted_labels"])
        print(f"{split}: {len(r['labels'])} aligned samples; model agreement={agreement:.2%}")
